# Step 05 — HMM estimation panel (region fixed, minimum 3 observed years)

**Output:** one streaming trip-level Parquet file with 39 variables, including categorical `regional_cd`.

**Region fix:** Read the exact source field `Region_Cd` from `household_demographics_2008_2017.tsv` and output it as `regional_cd` (Nielsen regions 1–9). The annual covariate Parquet does **not** need a region variable.

**Sample:** Keep households with at least 3 distinct years containing an observed shopping trip in 2008–2017. The years need not be consecutive.

**Speed:** The first run scans only `hhid` from each yearly trip Parquet to construct a reusable household-year tenure cache. Subsequent runs use the cache if input file paths, sizes, and timestamps are unchanged. One prejoined household-year lookup is used for the output streaming pass, with incremental progress logging.

**Safety:** The incomplete output is written to `.partial.parquet` and atomically replaces the final file only after successful validation. Restricted data and caches remain in `healthy_purchase_data`, outside GitHub. Run the cells in order. 

**Moving:** The `moving` event is taken unchanged from the existing `hmm_annual_covariates_income_imputed.parquet` household-year input. Step 05 does not infer or recode moving from ZIP codes.


In [ ]:
from pathlib import Path
from time import perf_counter
import gc
import json
import duckdb
import pyarrow as pa
import pyarrow.parquet as pq
import pyarrow.compute as pc
import pandas as pd

DATA_ROOT = Path(r"C:/Users/rwen13/Documents/healthy_purchase_data")
MASTER = DATA_ROOT / "processed/hmm_panel/hmm_master"
TRIP_DIR = MASTER / "hmm_trips"
ANNUAL_FILE = MASTER / "hmm_annual_covariates_income_imputed.parquet"
DEMO_FILE = DATA_ROOT / "processed/demographics/household_demographics_2008_2017.tsv"
INITIAL_FILE = MASTER / "hmm_initial_fresh_fv_share_2007.parquet"
OUTPUT_DIR = DATA_ROOT / "processed/hmm_input"
OUTPUT_FILE = OUTPUT_DIR / "hmm_estimation_panel.parquet"
PARTIAL_FILE = OUTPUT_DIR / "hmm_estimation_panel.partial.parquet"
AUDIT_DIR = OUTPUT_DIR / "validation"

# Set to True only when you explicitly intend to replace an existing final file.
OVERWRITE_EXISTING = False
REBUILD_TENURE_CACHE = False  # Force recreation of observed household-year keys.
MEMORY_LIMIT = "4GB"
THREADS = 4
BATCH_ROWS = 250_000  # Streaming batches; decrease to 100_000 if memory is tight.
COMPRESSION = "snappy"  # Faster to write than ZSTD; larger output.

EMISSIONS = ["fv_y", "milk_y", "yogurt_y", "cereal_y", "snack_y", "beverage_y"]
EVENTS = ["event_new_child", "getting_married", "getting_divorce_or_separation",
          "moving", "getting_job", "retirement", "other_unemployment"]
SPENDING = ["fresh_spending", "non_fresh_spending", "skimmed_milk_spending",
            "low_fat_milk_spending", "other_milk_spending", "skimmed_yogurt_spending",
            "low_fat_yogurt_spending", "other_yogurt_spending",
            "low_sugar_cereal_spending", "medium_sugar_cereal_spending", "high_sugar_cereal_spending",
            "healthy_snack_spending", "unhealthy_snack_spending",
            "healthy_beverage_spending", "unhealthy_beverage_spending"]

for p in [ANNUAL_FILE, DEMO_FILE, INITIAL_FILE]:
    if not p.is_file():
        raise FileNotFoundError(f"Missing required input: {p}")
trip_files = sorted(TRIP_DIR.glob("panel_year=*/trips.parquet"))
if not trip_files:
    raise FileNotFoundError(f"No trip partitions found at {TRIP_DIR}")
trip_by_year = {int(p.parent.name.split("=", 1)[1]): p for p in trip_files}
if set(trip_by_year) != set(range(2008, 2018)) or len(trip_files) != 10:
    raise ValueError(f"Expected exactly 2008–2017 trip files; got {sorted(trip_by_year)}")
if OUTPUT_FILE.exists() and not OVERWRITE_EXISTING:
    raise FileExistsError(f"Final output exists: {OUTPUT_FILE}. Set OVERWRITE_EXISTING=True to replace it.")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
AUDIT_DIR.mkdir(parents=True, exist_ok=True)
(DATA_ROOT / "temp").mkdir(parents=True, exist_ok=True)

# Never remove the final output before a replacement is fully created.
if PARTIAL_FILE.exists():
    print(f"Deleting leftover INCOMPLETE temporary file: {PARTIAL_FILE}", flush=True)
    PARTIAL_FILE.unlink()

def qpath(path):
    return "'" + str(path).replace(chr(92), "/").replace("'", "''") + "'"

con = duckdb.connect(database=":memory:")
con.execute(f"SET memory_limit = '{MEMORY_LIMIT}'")
con.execute(f"SET threads = {THREADS}")
con.execute(f"SET temp_directory = {qpath(DATA_ROOT / 'temp')}")
print(f"Ready: {len(trip_files)} yearly input files; {BATCH_ROWS:,} rows per streaming batch.", flush=True)

## 1. Create compact household-year lookup tables

First, read annual covariates and household demographics, mapping **`Region_Cd` → `regional_cd`**. Then identify actual observed trip years using a reusable cache. Restrict to households with at least 3 distinct trip years and prejoin regional/demographic/annual covariates once, instead of joining them separately for every trip. Validate key uniqueness and eligible household-year coverage before streaming.

The `moving` variable is passed through from the preconstructed annual covariates; no ZIP-based move detection is performed.

The interstate-moving variant computes `moving` from adjacent-year `Fips_State_Cd` changes in demographics before household tenure filtering. It records disagreements with upstream moving in a local audit.


In [ ]:
t0 = perf_counter()
# Fail fast if the exact region field is missing; never silently create an empty region.
with DEMO_FILE.open("r", encoding="utf-8-sig") as fh:
    demographic_columns = fh.readline().rstrip("\r\n").split("\t")
for required in ("Region_Cd", "Fips_State_Cd"):
    if required not in demographic_columns:
        raise ValueError(f"Required {required} missing from {DEMO_FILE}; available columns: {demographic_columns}")
print("[1/5] Loading demographics (including Region_Cd)...", flush=True)
# The data have many columns: never supply a truncated four-column CSV schema.
# Validate each head's birth-derived age before taking the household-head mean.
# Out-of-range birth-derived ages are replaced with the corresponding Nielsen age-band midpoint.
# Open-ended age bands (<25 and 65+) use documented assumptions of 22 and 70 years.
# Missing/absent heads are not treated as age zero.
con.execute(f"""
    CREATE TEMP TABLE demo_age AS
    WITH raw AS (
        SELECT CAST(hhid AS VARCHAR) AS hhid,
               TRY_CAST(panel_year AS INTEGER) AS panel_year,
               TRY_CAST(male_head_age_from_birth AS DOUBLE) AS male_birth_age,
               TRY_CAST(female_head_age_from_birth AS DOUBLE) AS female_birth_age,
               TRY_CAST(Male_Head_Age AS INTEGER) AS male_age_code,
               TRY_CAST(Female_Head_Age AS INTEGER) AS female_age_code,
               TRY_CAST(NULLIF(TRIM("Region_Cd"), '') AS TINYINT) AS regional_cd,
               -- State FIPS is an identifier. Normalize to two digits and reject invalid codes.
               CASE WHEN regexp_full_match(TRIM("Fips_State_Cd"), '[0-9]{1,2}')
                         AND TRY_CAST(TRIM("Fips_State_Cd") AS INTEGER) IN
                             (1,2,4,5,6,8,9,10,11,12,13,15,16,17,18,19,20,21,22,
                              23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,
                              40,41,42,44,45,46,47,48,49,50,51,53,54,55,56)
                    THEN LPAD(TRIM("Fips_State_Cd"), 2, '0')
                    ELSE NULL END AS state_fips
        FROM read_csv({qpath(DEMO_FILE)}, delim='\t', header=true,
                      all_varchar=true, quote='', escape='', strict_mode=true)
    ), coded AS (
        SELECT *,
            CASE male_age_code
                WHEN 1 THEN 22.0 WHEN 2 THEN 27.0 WHEN 3 THEN 32.0
                WHEN 4 THEN 37.0 WHEN 5 THEN 42.0 WHEN 6 THEN 47.0
                WHEN 7 THEN 52.0 WHEN 8 THEN 59.5 WHEN 9 THEN 70.0
            END AS male_band_age,
            CASE female_age_code
                WHEN 1 THEN 22.0 WHEN 2 THEN 27.0 WHEN 3 THEN 32.0
                WHEN 4 THEN 37.0 WHEN 5 THEN 42.0 WHEN 6 THEN 47.0
                WHEN 7 THEN 52.0 WHEN 8 THEN 59.5 WHEN 9 THEN 70.0
            END AS female_band_age
        FROM raw
    ), selected AS (
        SELECT *,
               CASE WHEN male_birth_age BETWEEN 18 AND 110 THEN male_birth_age
                    ELSE male_band_age END AS male_age,
               CASE WHEN female_birth_age BETWEEN 18 AND 110 THEN female_birth_age
                    ELSE female_band_age END AS female_age,
               (male_birth_age IS NOT NULL AND NOT (male_birth_age BETWEEN 18 AND 110))
                   AS male_birth_age_invalid,
               (female_birth_age IS NOT NULL AND NOT (female_birth_age BETWEEN 18 AND 110))
                   AS female_birth_age_invalid
        FROM coded
    )
    SELECT hhid, panel_year, regional_cd, state_fips,
           CASE WHEN male_age IS NOT NULL AND female_age IS NOT NULL
                    THEN (male_age + female_age) / 2.0
                ELSE COALESCE(male_age, female_age) END AS age,
           male_birth_age_invalid, female_birth_age_invalid,
           CASE WHEN male_birth_age_invalid AND male_band_age IS NOT NULL THEN 1 ELSE 0 END
               + CASE WHEN female_birth_age_invalid AND female_band_age IS NOT NULL THEN 1 ELSE 0 END
               AS age_head_fallback_count
    FROM selected
""")
print(f"[1/5] Demographics loaded in {perf_counter()-t0:.1f}s", flush=True)
print("[2/5] Loading annual covariates...", flush=True)
con.execute(f"""
    CREATE TEMP TABLE annual_keyed AS
    SELECT CAST(hhid AS VARCHAR) AS hhid,
           CAST(panel_year AS INTEGER) AS panel_year,
-- Modeling convention: missing annual transition event indicators are treated as no event (0).
           {', '.join('COALESCE(TRY_CAST(' + v + ' AS TINYINT), 0)::TINYINT AS ' + v for v in EVENTS)},
           TRY_CAST(K_1 AS TINYINT) AS K_1,
           TRY_CAST(income10k_imputed AS DOUBLE) AS income10k_imputed,
           TRY_CAST(family_size_numeric AS DOUBLE) AS family_size_numeric
    FROM read_parquet({qpath(ANNUAL_FILE)})
""")

print(f"[2/5] Covariates loaded in {perf_counter()-t0:.1f}s", flush=True)
for name in ("annual_keyed", "demo_age"):
    duplicate = con.execute(f"""
        SELECT hhid, panel_year, COUNT(*) AS n
        FROM {name} GROUP BY hhid, panel_year HAVING COUNT(*) > 1 LIMIT 1
    """).fetchone()
    if duplicate:
        raise ValueError(f"Duplicate household-year key in {name}: {duplicate}")

coverage = con.execute("""
    SELECT COUNT(*) AS annual_rows,
           COUNT(d.hhid) AS matched_age_rows,
           COUNT(*) FILTER (WHERE d.age IS NULL) AS missing_age,
           COUNT(*) FILTER (WHERE d.age NOT BETWEEN 18 AND 110) AS invalid_age,
           COUNT(*) FILTER (WHERE a.income10k_imputed IS NULL) AS missing_income
    FROM annual_keyed a LEFT JOIN demo_age d USING (hhid, panel_year)
""").fetchone()
print("Annual rows | age matches | missing age | invalid age | missing income:", coverage, flush=True)
# Lightweight age QA: report source-age anomalies and category-based corrections.
age_fallback_audit = con.execute("""
    SELECT COUNT(*) FILTER (WHERE male_birth_age_invalid OR female_birth_age_invalid)
               AS household_years_with_invalid_head_birth_age,
           COALESCE(SUM(age_head_fallback_count), 0) AS corrected_head_age_values,
           COUNT(*) FILTER (WHERE age IS NULL OR age NOT BETWEEN 18 AND 110)
               AS unresolved_household_years
    FROM demo_age
""").fetchone()
print("Age QA | anomalous HH-years | head values corrected | unresolved:", age_fallback_audit, flush=True)
region_qa = con.execute("""
    SELECT COUNT(*) FILTER (WHERE regional_cd IS NULL),
           COUNT(*) FILTER (WHERE regional_cd NOT BETWEEN 1 AND 9)
    FROM demo_age
""").fetchone()
print("Region QA | missing household-years | invalid region codes:", region_qa, flush=True)
if region_qa[1]:
    raise ValueError(f"Invalid Region_Cd values outside 1–9: {region_qa[1]}")

# Redefine moving as an interstate move using changes in state FIPS codes.
# Only *consecutive calendar years* with valid values in both years are comparable.
# Unobserved comparisons are encoded 0 in the existing binary event column,
# but counted separately in the audit (0 is not proof of no move).
# Use the complete demographics history before the >=3 trip-year restriction.
con.execute("""
    CREATE TEMP TABLE interstate_moving AS
    WITH previous AS (
        SELECT hhid, panel_year, state_fips,
               LAG(panel_year) OVER (PARTITION BY hhid ORDER BY panel_year) AS prior_year,
               LAG(state_fips) OVER (PARTITION BY hhid ORDER BY panel_year) AS prior_state
        FROM demo_age
    )
    SELECT hhid, panel_year,
           CASE WHEN prior_year = panel_year - 1
                 AND state_fips IS NOT NULL AND prior_state IS NOT NULL
                 AND state_fips <> prior_state THEN 1 ELSE 0 END::TINYINT AS moving_interstate,
           (prior_year = panel_year - 1
                 AND state_fips IS NOT NULL AND prior_state IS NOT NULL) AS state_pair_observed
    FROM previous
""")
# Audit only the small household-year source; this does not scan shopping trips.
interstate_audit = con.execute("""
    SELECT COUNT(*) AS matched_household_years,
           COUNT(*) FILTER (WHERE m.state_pair_observed) AS comparable_state_pairs,
           COUNT(*) FILTER (WHERE NOT m.state_pair_observed) AS unobservable_state_pairs,
           COUNT(*) FILTER (WHERE m.moving_interstate = 1) AS interstate_moves,
           COUNT(*) FILTER (WHERE a.moving = 1) AS original_moving_events,
           COUNT(*) FILTER (WHERE m.moving_interstate <> a.moving) AS disagreements
    FROM annual_keyed a
    INNER JOIN interstate_moving m USING (hhid, panel_year)
""").fetchone()
print("Interstate moving QA | household-years | comparable | unobservable | "
      "interstate moves | original moving | disagreements:", interstate_audit, flush=True)

print("[3/5] Preparing observed-year eligibility cache...", flush=True)
# Source path, size and mtime define a conservative cache signature.
CACHE_DIR = DATA_ROOT / "temp" / "step05_tenure_cache"
CACHE_DIR.mkdir(parents=True, exist_ok=True)
CACHE_FILE = CACHE_DIR / "observed_household_year.parquet"
CACHE_MANIFEST = CACHE_DIR / "observed_household_year_manifest.json"
input_signature = [
    {"year": year, "path": str(path.resolve()), "size": path.stat().st_size,
     "mtime_ns": path.stat().st_mtime_ns}
    for year, path in sorted(trip_by_year.items())
]
cache_valid = False
if not REBUILD_TENURE_CACHE and CACHE_FILE.is_file() and CACHE_MANIFEST.is_file():
    try:
        cache_valid = json.loads(CACHE_MANIFEST.read_text(encoding="utf-8")) == input_signature
    except (OSError, ValueError):
        cache_valid = False

if cache_valid:
    print("[3/5] Cache HIT: skipping trip-year ID scan", flush=True)
    con.execute(f"""
        CREATE TEMP TABLE observed_household_year AS
        SELECT CAST(hhid AS VARCHAR) AS hhid,
               CAST(panel_year AS INTEGER) AS panel_year
        FROM read_parquet({qpath(CACHE_FILE)})
    """)
else:
    print("[3/5] Cache MISS: scanning only hhid by year (one-time cost)", flush=True)
    con.execute("CREATE TEMP TABLE observed_household_year (hhid VARCHAR, panel_year INTEGER)")
    for year, path in sorted(trip_by_year.items()):
        scan_start = perf_counter()
        print(f"[3/5] {year}: scan household IDs...", flush=True)
        con.execute(f"""
            INSERT INTO observed_household_year
            SELECT DISTINCT CAST(hhid AS VARCHAR), {year}::INTEGER
            FROM read_parquet({qpath(path)})
        """)
        print(f"[3/5] {year}: done in {perf_counter()-scan_start:.1f}s", flush=True)

invalid_observed = con.execute("""
    SELECT hhid, panel_year FROM observed_household_year
    WHERE hhid IS NULL OR panel_year NOT BETWEEN 2008 AND 2017 LIMIT 1
""").fetchone()
if invalid_observed:
    raise ValueError(f"Invalid observed household-year: {invalid_observed}")
duplicate_observed = con.execute("""
    SELECT hhid, panel_year FROM observed_household_year
    GROUP BY 1, 2 HAVING COUNT(*) > 1 LIMIT 1
""").fetchone()
if duplicate_observed:
    raise ValueError(f"Duplicate observed household-year: {duplicate_observed}")
if not cache_valid:
    partial_cache = CACHE_DIR / "observed_household_year.partial.parquet"
    con.execute(f"COPY observed_household_year TO {qpath(partial_cache)} (FORMAT PARQUET, COMPRESSION SNAPPY)")
    partial_cache.replace(CACHE_FILE)
    partial_manifest = CACHE_DIR / "observed_household_year_manifest.partial.json"
    partial_manifest.write_text(json.dumps(input_signature, indent=2), encoding="utf-8")
    partial_manifest.replace(CACHE_MANIFEST)
    print("[3/5] Wrote exact household-year cache", flush=True)

con.execute("""
    CREATE TEMP TABLE household_panel_tenure AS
    SELECT hhid, COUNT(*)::INTEGER AS observed_years
    FROM observed_household_year
    GROUP BY hhid
""")
tenure_summary = con.execute("""
    SELECT observed_years, COUNT(*)::BIGINT AS households
    FROM household_panel_tenure GROUP BY 1 ORDER BY 1
""").fetchall()
if not any(years >= 3 for years, households in tenure_summary):
    raise ValueError("No households have at least three observed shopping years")
print("Tenure audit | observed years and household counts:", tenure_summary, flush=True)

# One row per eligible household-year; only one join is required per output trip.
print("[4/5] Building combined eligible household-year lookup...", flush=True)
con.execute("""
    CREATE TEMP TABLE model_lookup AS
    SELECT a.hhid, a.panel_year,
           a.event_new_child, a.getting_married, a.getting_divorce_or_separation,
           m.moving_interstate AS moving, a.getting_job, a.retirement, a.other_unemployment,
           a.K_1, a.income10k_imputed, a.family_size_numeric,
           d.age, d.regional_cd
    FROM annual_keyed a
    INNER JOIN demo_age d USING (hhid, panel_year)
    INNER JOIN interstate_moving m USING (hhid, panel_year)
    INNER JOIN household_panel_tenure t USING (hhid)
    WHERE t.observed_years >= 3
""")
# Check the small observed household-year key set for dropped covariates.
missing_lookup = con.execute("""
    SELECT o.hhid, o.panel_year
    FROM observed_household_year o
    INNER JOIN household_panel_tenure t USING (hhid)
    ANTI JOIN model_lookup m USING (hhid, panel_year)
    WHERE t.observed_years >= 3
    LIMIT 1
""").fetchone()
if missing_lookup:
    raise ValueError(f"Eligible household-year missing annual/demo covariates: {missing_lookup}")
duplicated_lookup = con.execute("""
    SELECT hhid, panel_year FROM model_lookup
    GROUP BY 1, 2 HAVING COUNT(*) > 1 LIMIT 1
""").fetchone()
if duplicated_lookup:
    raise ValueError(f"Duplicate eligible household-year in model lookup: {duplicated_lookup}")
con.execute("DROP TABLE observed_household_year")
con.execute("DROP TABLE annual_keyed")
con.execute("DROP TABLE demo_age")
con.execute("DROP TABLE interstate_moving")
con.execute("DROP TABLE household_panel_tenure")
print(f"[4/5] Lookup ready after {perf_counter()-t0:.1f}s", flush=True)


if coverage[0] != coverage[1] or any(coverage[i] for i in (2, 3, 4)):
    raise ValueError("Annual covariates fail age or income coverage checks.")

pi_values = con.execute(f"""
    SELECT TRY_CAST(initial_fresh_fv_share_2007 AS DOUBLE)
    FROM read_parquet({qpath(INITIAL_FILE)})
""").fetchall()
if len(pi_values) != 1 or pi_values[0][0] is None or not (0 < pi_values[0][0] < 1):
    raise ValueError(f"Expected exactly one common initialization share in (0, 1): {pi_values}")
PI = float(pi_values[0][0])

# Pre-check source columns without scanning any trip rows.
required_trip = set(["hhid", "panel_year", "trip_id", "purchase_date", *EMISSIONS, *SPENDING, "food_spend"])
expected_rows_by_year = {}
for year, trip_file in trip_by_year.items():
    pf = pq.ParquetFile(trip_file)
    missing = required_trip - set(pf.schema_arrow.names)
    if missing:
        raise ValueError(f"Missing trip columns for {year}: {sorted(missing)}")
    expected_rows_by_year[year] = pf.metadata.num_rows
print(f"Lookup validated in {perf_counter()-t0:.1f}s; common pi start={PI:.8f}", flush=True)
print(f"Source trip rows: {sum(expected_rows_by_year.values()):,}", flush=True)

## 2. Single-join streaming output

Only eligible households are included. The query joins each yearly trip partition to `model_lookup` once, and writes Arrow batches directly to the output Parquet. A cached household-year eligibility table eliminates repeated full trip aggregations on subsequent runs. Counts and missing-region indicators are accumulated while streaming.


In [ ]:
selected_emissions = ", ".join(f"TRY_CAST(t.{v} AS TINYINT) AS {v}" for v in EMISSIONS)
selected_events = ", ".join(f"m.{v} AS {v}" for v in EVENTS)
selected_spending = ", ".join(f"TRY_CAST(t.{v} AS DOUBLE) AS {v}" for v in SPENDING)

# Accumulate small audit counters during the same streaming pass.
rows_written = 0
rows_by_year = []
missing_events = {v: 0 for v in EVENTS}  # Post-fill NULL audit: must remain zero.
bad_emissions = {v: 0 for v in EMISSIONS}
missing_food_spend = 0
missing_region_trips = 0
writer = None
started = perf_counter()

try:
    for year in sorted(trip_by_year):
        file = trip_by_year[year]
        source_rows = expected_rows_by_year[year]
        year_start = perf_counter()
        year_rows = 0
        print(f"[{year}] START | source rows={source_rows:,}", flush=True)

        # No ORDER BY: trip order is not part of the HMM data contract.
        sql = f"""
            SELECT CAST(t.hhid AS VARCHAR) AS hhid,
                   CAST(t.panel_year AS INTEGER) AS panel_year,
                   CAST(t.trip_id AS VARCHAR) AS trip_id,
                   CAST(t.purchase_date AS DATE) AS purchase_date,
                   {selected_emissions}, {selected_events},
                   m.K_1 AS K_1,
                   m.income10k_imputed AS income10k_imputed,
                   m.family_size_numeric AS family_size_numeric,
                   m.age AS age,
                   m.regional_cd AS regional_cd,
                   {selected_spending},
                   TRY_CAST(t.food_spend AS DOUBLE) AS food_spend,
                   {PI:.17g}::DOUBLE AS initial_fresh_fv_share_2007
            FROM read_parquet({qpath(file)}) t
            INNER JOIN model_lookup m
              ON CAST(t.hhid AS VARCHAR) = m.hhid
             AND CAST(t.panel_year AS INTEGER) = m.panel_year
        """
        # DuckDB streams record batches; nothing materializes 37M trip rows in RAM.
        reader = con.execute(sql).fetch_record_batch(rows_per_batch=BATCH_ROWS)
        for batch_no, batch in enumerate(reader, start=1):
            if writer is None:
                writer = pq.ParquetWriter(
                    str(PARTIAL_FILE), batch.schema, compression=COMPRESSION,
                    use_dictionary=True, write_statistics=True
                )
            if not batch.schema.equals(writer.schema, check_metadata=False):
                raise ValueError(f"Inconsistent output schema detected for year {year}")

            # Lightweight batch-level validation, while data are already loaded.
            for name in EMISSIONS:
                arr = batch.column(batch.schema.get_field_index(name))
                bad_emissions[name] += arr.null_count
                bounds = pc.min_max(arr).as_py()
                if bounds['min'] is not None and bounds['min'] < 0:
                    raise ValueError(f"Invalid negative {name} in year {year}")
                if bounds['max'] is not None and bounds['max'] > (3 if name == 'cereal_y' else 2):
                    raise ValueError(f"Invalid {name} > allowed range in year {year}")
            for name in EVENTS:
                arr = batch.column(batch.schema.get_field_index(name))
                missing_events[name] += arr.null_count
                if arr.null_count:
                    raise ValueError(f"Missing {name} remained after zero-filling in year {year}")
            for name in ("age", "income10k_imputed"):
                arr = batch.column(batch.schema.get_field_index(name))
                if arr.null_count:
                    raise ValueError(f"Missing {name} in {year}: {arr.null_count} trip rows")
            missing_region_trips += batch.column(batch.schema.get_field_index("regional_cd")).null_count
            amount = batch.column(batch.schema.get_field_index("food_spend"))
            missing_food_spend += amount.null_count  # Preserve NA; never silently fill with zero.
            minimum = pc.min(amount).as_py()
            if minimum is not None and minimum < 0:
                raise ValueError(f"Negative food_spend detected in {year}")
            if batch.column(batch.schema.get_field_index("panel_year")).null_count:
                raise ValueError(f"Missing panel_year in {year}")

            writer.write_table(pa.Table.from_batches([batch]), row_group_size=BATCH_ROWS)
            year_rows += batch.num_rows
            rows_written += batch.num_rows
            if batch_no % 5 == 0:
                print(f"[{year}] {year_rows:,} trip rows written", flush=True)

        if year_rows > source_rows:
            raise ValueError(f"Year {year}: more output trips than input trips")
        rows_by_year.append({"panel_year": year, "source_trips": source_rows,
                             "excluded_tenure_trips": source_rows-year_rows,
                             "output_trips": year_rows,
                             "elapsed_seconds": round(perf_counter()-year_start, 2)})
        print(f"[{year}] COMPLETE | {year_rows:,} trips | {perf_counter()-year_start:.1f}s", flush=True)
        gc.collect()

finally:
    if writer is not None:
        writer.close()  # On failure, the .partial file is NOT published.

if any(bad_emissions.values()):
    raise ValueError(f"Missing or invalid emission outcomes: {bad_emissions}")
print(f"STREAMING BUILD COMPLETE: {rows_written:,} trips, {perf_counter()-started:.1f}s", flush=True)

## 3. Validate output and publish

Check output row and column counts from the Parquet footer. Write source-vs-retained trip audits and the household tenure distribution. All audits remain outside GitHub. The final file is atomically replaced only after validation.


In [ ]:
metadata = pq.read_metadata(PARTIAL_FILE)
expected_total = rows_written
if metadata.num_rows != expected_total or metadata.num_rows != rows_written:
    raise ValueError(f"Parquet footer row count mismatch: {metadata.num_rows:,} vs {expected_total:,}")
if metadata.num_columns != 39:
    raise ValueError(f"Expected 39 variables, found {metadata.num_columns}")

# These are small, non-sensitive aggregate checks, kept outside the repository.
pd.DataFrame(rows_by_year).to_csv(AUDIT_DIR / "step05_streaming_rows_by_year.csv", index=False)
# Compare original source moving with interstate moves at the household-year level.
pd.DataFrame([dict(zip([
    "matched_household_years", "comparable_state_pairs", "unobservable_state_pairs",
    "interstate_moves", "original_moving_events", "disagreements"
], interstate_audit))]).to_csv(AUDIT_DIR / "step05_interstate_moving_audit.csv", index=False)
pd.DataFrame(tenure_summary, columns=["observed_years", "households"]).assign(
    status=lambda df: df["observed_years"].map(lambda years: "retained" if years >= 3 else "excluded")
).to_csv(AUDIT_DIR / "step05_household_tenure_audit.csv", index=False)
pd.DataFrame([{"variable": "regional_cd", "missing_trip_rows": missing_region_trips}]).to_csv(
    AUDIT_DIR / "step05_streaming_region_audit.csv", index=False
)
pd.DataFrame([{"variable": v, "missing_trip_rows": missing_events[v]} for v in EVENTS]).to_csv(
    AUDIT_DIR / "step05_streaming_missing_life_events.csv", index=False
)
pd.DataFrame([{"variable": v, "missing_or_invalid": bad_emissions[v]} for v in EMISSIONS] +
             [{"variable": "food_spend", "missing_or_invalid": missing_food_spend}]).to_csv(
    AUDIT_DIR / "step05_streaming_value_audit.csv", index=False
)

# Replace only after the new Parquet is closed and validated.
PARTIAL_FILE.replace(OUTPUT_FILE)
print(f"SAVED: {OUTPUT_FILE}", flush=True)
print(f"ROWS: {metadata.num_rows:,} | COLUMNS: {metadata.num_columns} | SIZE: {OUTPUT_FILE.stat().st_size/1024**3:.2f} GiB", flush=True)
print(f"Missing food_spend (retained as NULL): {missing_food_spend:,}", flush=True)
print(f"Missing regional_cd (retained as NULL): {missing_region_trips:,}", flush=True)
print("Null life-event trip counts:", missing_events, flush=True)
con.close()

### Notes for model estimation

- This is a **single file**, as requested; repeated covariates do not imply trip-level state transitions.
- `hhid` and `trip_id` remain strings. Emission outcomes are small integers; category spending is numeric.
- Missing life-event / transition covariate values are set to `0` in the estimation panel (interpreted as no recorded event); the original annual input is not modified. This is an explicit modeling assumption, not evidence that the event did not occur.
- The common 2007 fresh-FV share is an **initialization value** for the HMM's common Healthy-state probability, not a fixed latent-state probability.
- Because the output is not globally sorted, **sort internally by household/year/trip when building HMM arrays**. Handle calendar-year gaps explicitly.
- `food_spend` may be missing for some unmatched trips; it is not filled with zero.
- A row count comparison verifies no trips were lost or multiplied. It does **not** prove trip-key uniqueness or purchase-level Magnet monetary reconciliation; those remain Step 03/02 responsibilities.
- `regional_cd` comes from `Region_Cd` in household-year demographics and is coded 1–9. Treat as categorical in HMM estimation, not as a continuous regressor. Missing values remain NULL and are audited.
- The 3-year cutoff uses distinct years with observed shopping trips, not merely the presence of demographic records. Nonconsecutive years count.
- The tenure cache is stored locally under `healthy_purchase_data/temp/step05_tenure_cache`. Set `REBUILD_TENURE_CACHE=True` to force an exact rebuild.

- `moving` comes directly from `hmm_annual_covariates_income_imputed.parquet`, without redefinition in Step 05.

- In this variant, `moving` means an interstate move (valid state FIPS differs from the preceding consecutive calendar year). Within-state moves are coded 0, and missing/nonconsecutive comparisons are also coded 0 but separately audited. The upstream annual covariates Parquet remains unchanged.
